# GraphRAG: Knowledge Graphs for Smarter Document QA

**Based on:** Edge et al. (2024), ["From Local to Global: A Graph RAG Approach to Query-Focused Summarisation"](https://arxiv.org/abs/2404.16130), arXiv:2404.16130.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/graphrag_from_scratch.ipynb)

This notebook walks through the full GraphRAG pipeline from scratch:

1. Text chunking and entity/relationship extraction
2. Knowledge graph construction
3. GIF animation of graph construction
4. Community detection (Louvain)
5. Community report generation
6. Global search (map-reduce over all communities)
7. Local search (entity-anchored retrieval)
8. LlamaIndex GraphRAG V2 comparison
9. Exercises

**Requirements:** An `OPENAI_API_KEY` environment variable.

## Cell 1: Setup and Imports

In [ ]:
# Install dependencies
# !pip install openai networkx python-louvain matplotlib pillow

import os
import json
import textwrap
from collections import defaultdict

import networkx as nx
import community as community_louvain  # python-louvain
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import PillowWriter
from openai import OpenAI

# Set your API key here or via environment variable
# os.environ["OPENAI_API_KEY"] = "sk-..."

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Setup complete.")

## Cell 2: Corpus Definition

We use a hardcoded 8-paragraph ML corpus so no file downloads are needed.
Each paragraph is one chunk. The paper recommends 1,200 tokens per chunk;
these paragraphs are shorter but sufficient to demonstrate the pipeline.

In [ ]:
CORPUS = [
    """Transformer models use self-attention to relate all positions in a sequence
    simultaneously. The attention mechanism computes query, key, and value projections,
    then applies scaled dot-product attention. BERT introduced bidirectional
    pre-training, while GPT models use causal (left-to-right) attention masks.""",

    """Large language models are typically trained with the next-token prediction
    objective on massive text corpora. GPT-4 and Claude use reinforcement learning
    from human feedback (RLHF) to align model outputs with human preferences.
    Constitutional AI from Anthropic adds a self-critique step before RLHF.""",

    """Retrieval-augmented generation (RAG) combines a dense retrieval system with
    a generative language model. A query is encoded into an embedding, the nearest
    chunks are retrieved from a vector store such as FAISS or Pinecone, and the
    context is prepended to the LLM prompt. RAG reduces hallucination and allows
    knowledge to be updated without retraining.""",

    """Knowledge graphs represent facts as (subject, predicate, object) triples.
    Neo4j is a popular property graph database. SPARQL is used to query RDF graphs.
    Entity linking maps surface mentions to canonical graph nodes, and relation
    extraction populates the graph from unstructured text.""",

    """Quantisation reduces model size by representing weights in lower precision.
    INT8 and INT4 are common targets. GPTQ applies one-shot post-training
    quantisation using approximate second-order information. AWQ activates
    weight quantisation to preserve salient weights. Both methods keep accuracy
    close to the full-precision baseline.""",

    """The KV cache stores the key and value tensors from previous attention
    computations so they need not be recomputed at each decoding step. This trades
    memory for compute. For a 70B-parameter model with long contexts, the KV cache
    can exceed the size of the model weights themselves, making it a primary
    bottleneck for batch throughput.""",

    """Community detection algorithms partition graph nodes into groups with dense
    internal connections. The Louvain algorithm optimises modularity greedily.
    Leiden improves on Louvain by guaranteeing well-connected communities and
    avoiding poorly connected splits. Both algorithms are widely used for
    social network analysis and knowledge graph clustering.""",

    """Vector databases such as Weaviate, Qdrant, and Chroma store dense embeddings
    and support approximate nearest-neighbour search via HNSW or IVF indexing.
    They underpin semantic search, recommendation engines, and RAG pipelines.
    Hybrid search combines dense retrieval with sparse BM25 scoring.""",
]

print(f"Corpus: {len(CORPUS)} chunks")
for i, chunk in enumerate(CORPUS):
    word_count = len(chunk.split())
    print(f"  Chunk {i+1}: {word_count} words")

## Cell 3: Entity and Relationship Extraction

For each chunk we ask `gpt-4o-mini` to return a structured JSON payload with:
- **Entities:** name, type, and description
- **Relationships:** source, target, description, and numeric strength

The paper calls for multiple "gleaning" passes per chunk to improve recall.
We do one pass here; see the gleaning extension at the bottom of this notebook.

In [ ]:
EXTRACTION_PROMPT = """You are an information-extraction assistant.
Given a text passage, extract all named entities and the relationships between them.

Return ONLY valid JSON with this exact structure:
{
  "entities": [
    {"name": "...", "type": "...", "description": "..."}
  ],
  "relationships": [
    {"source": "...", "target": "...", "description": "...", "strength": 0.0}
  ]
}

Rules:
- "type" should be one of: MODEL, TECHNIQUE, CONCEPT, TOOL, ALGORITHM, METRIC
- "strength" is a float from 0 to 1 indicating how strongly the entities are related
- Entity names should be canonical (e.g. "GPT-4" not "it")
- Only include relationships explicitly supported by the text
"""


def extract_entities_and_relationships(chunk: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": EXTRACTION_PROMPT},
            {"role": "user", "content": f"Text:\n{chunk}"},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


print("Extracting entities and relationships from each chunk...")
extractions = [extract_entities_and_relationships(chunk) for chunk in CORPUS]

total_entities = 0
total_rels = 0
for i, ext in enumerate(extractions):
    n_ent = len(ext.get("entities", []))
    n_rel = len(ext.get("relationships", []))
    total_entities += n_ent
    total_rels += n_rel
    print(f"  Chunk {i+1}: {n_ent} entities, {n_rel} relationships")

print(f"\nTotal before merging: {total_entities} entities, {total_rels} relationships")

## Cell 4: Graph Construction

We merge all extracted entities and relationships into a single NetworkX graph:
- Duplicate entity names have their descriptions concatenated.
- Duplicate edges accumulate strength (capped at 1.0).

In [ ]:
G = nx.Graph()

for ext in extractions:
    for entity in ext.get("entities", []):
        name = entity["name"]
        if name not in G:
            G.add_node(name, type=entity["type"], description=entity["description"])
        else:
            existing = G.nodes[name].get("description", "")
            if entity["description"] not in existing:
                G.nodes[name]["description"] = existing + " " + entity["description"]

    for rel in ext.get("relationships", []):
        src, tgt = rel["source"], rel["target"]
        if src in G and tgt in G:
            if G.has_edge(src, tgt):
                G[src][tgt]["strength"] = min(
                    1.0, G[src][tgt]["strength"] + rel["strength"]
                )
            else:
                G.add_edge(
                    src, tgt,
                    description=rel["description"],
                    strength=rel["strength"]
                )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Quick static visualisation
fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=1.5)

# Colour by entity type
type_colours = {
    "MODEL": "#4e79a7",
    "TECHNIQUE": "#f28e2b",
    "CONCEPT": "#59a14f",
    "TOOL": "#e15759",
    "ALGORITHM": "#76b7b2",
    "METRIC": "#edc948",
}
node_colours = [
    type_colours.get(G.nodes[n].get("type", "CONCEPT"), "#bab0ac")
    for n in G.nodes()
]
edge_weights = [G[u][v].get("strength", 0.5) for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=600, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, width=[w * 2 for w in edge_weights], alpha=0.4, ax=ax)

legend_patches = [
    mpatches.Patch(color=c, label=t) for t, c in type_colours.items()
]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8)
ax.set_title("GraphRAG Knowledge Graph", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 5: GIF Animation of Graph Construction

This cell generates the animated GIF showing:
- Frames 1-4: Entity nodes appearing one at a time
- Frames 5-8: Edges being drawn between entities
- Frames 9-11: Community colour clusters appearing
- Frame 12: Final state with community labels overlaid

The GIF is saved to `graphrag-knowledge-graph-document-qa-graph-build.gif`.

In [ ]:
import matplotlib.animation as animation

# Run community detection first so we have colours for later frames
partition_for_gif = community_louvain.best_partition(G, weight="strength", random_state=42)
comm_ids = sorted(set(partition_for_gif.values()))
palette = plt.cm.Set2.colors  # type: ignore[attr-defined]
comm_colour_map = {c: palette[i % len(palette)] for i, c in enumerate(comm_ids)}

nodes_list = list(G.nodes())
edges_list = list(G.edges())
pos_gif = nx.spring_layout(G, seed=42, k=1.5)

# Divide nodes and edges into 4 batches each for animation
def batches(lst, n):
    k = max(1, len(lst) // n)
    return [lst[i:i+k] for i in range(0, len(lst), k)]

node_batches = batches(nodes_list, 4)
edge_batches = batches(edges_list, 4)

fig_gif, ax_gif = plt.subplots(figsize=(10, 7))

def draw_frame(frame_idx):
    ax_gif.clear()
    ax_gif.axis("off")

    if frame_idx < 4:
        # Frames 0-3: nodes appearing batch by batch
        visible_nodes = []
        for b in range(frame_idx + 1):
            if b < len(node_batches):
                visible_nodes.extend(node_batches[b])
        subG = G.subgraph(visible_nodes)
        sub_pos = {n: pos_gif[n] for n in visible_nodes}
        node_cols = [type_colours.get(G.nodes[n].get("type", "CONCEPT"), "#bab0ac") for n in visible_nodes]
        nx.draw_networkx_nodes(subG, sub_pos, node_color=node_cols, node_size=500, alpha=0.85, ax=ax_gif)
        nx.draw_networkx_labels(subG, sub_pos, font_size=6, ax=ax_gif)
        ax_gif.set_title(f"Step 1: Entity nodes appearing ({len(visible_nodes)}/{len(nodes_list)})", fontsize=12)

    elif frame_idx < 8:
        # Frames 4-7: all nodes + edges appearing batch by batch
        edge_frame = frame_idx - 4
        visible_edges = []
        for b in range(edge_frame + 1):
            if b < len(edge_batches):
                visible_edges.extend(edge_batches[b])
        node_cols = [type_colours.get(G.nodes[n].get("type", "CONCEPT"), "#bab0ac") for n in G.nodes()]
        nx.draw_networkx_nodes(G, pos_gif, node_color=node_cols, node_size=500, alpha=0.85, ax=ax_gif)
        nx.draw_networkx_labels(G, pos_gif, font_size=6, ax=ax_gif)
        if visible_edges:
            subG_e = nx.Graph()
            subG_e.add_nodes_from(G.nodes())
            subG_e.add_edges_from(visible_edges)
            nx.draw_networkx_edges(subG_e, pos_gif, alpha=0.5, width=1.5, ax=ax_gif)
        ax_gif.set_title(f"Step 2: Edges being drawn ({len(visible_edges)}/{len(edges_list)})", fontsize=12)

    elif frame_idx < 11:
        # Frames 8-10: community colours phasing in
        comm_frame = frame_idx - 8
        visible_comms = comm_ids[: comm_frame + 1]
        node_cols = []
        for n in G.nodes():
            c = partition_for_gif[n]
            if c in visible_comms:
                node_cols.append(comm_colour_map[c])
            else:
                node_cols.append("#cccccc")
        nx.draw_networkx_nodes(G, pos_gif, node_color=node_cols, node_size=500, alpha=0.9, ax=ax_gif)
        nx.draw_networkx_labels(G, pos_gif, font_size=6, ax=ax_gif)
        nx.draw_networkx_edges(G, pos_gif, alpha=0.3, width=1.0, ax=ax_gif)
        ax_gif.set_title(f"Step 3: Community colours appearing ({len(visible_comms)}/{len(comm_ids)} communities)", fontsize=12)

    else:
        # Frame 11: final state with community labels
        node_cols = [comm_colour_map[partition_for_gif[n]] for n in G.nodes()]
        nx.draw_networkx_nodes(G, pos_gif, node_color=node_cols, node_size=550, alpha=0.95, ax=ax_gif)
        nx.draw_networkx_labels(G, pos_gif, font_size=6, ax=ax_gif)
        nx.draw_networkx_edges(G, pos_gif, alpha=0.35, width=1.0, ax=ax_gif)
        # Community label overlays
        comm_centroids = {}
        for comm_id in comm_ids:
            members = [n for n, c in partition_for_gif.items() if c == comm_id]
            xs = [pos_gif[m][0] for m in members]
            ys = [pos_gif[m][1] for m in members]
            comm_centroids[comm_id] = (sum(xs) / len(xs), sum(ys) / len(ys))
        for comm_id, (cx, cy) in comm_centroids.items():
            ax_gif.text(
                cx, cy + 0.12,
                f"Community {comm_id}",
                fontsize=9, fontweight="bold",
                ha="center", va="center",
                color=comm_colour_map[comm_id],
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7)
            )
        ax_gif.set_title("Step 4: Final graph with community clusters", fontsize=12)


anim = animation.FuncAnimation(
    fig_gif, draw_frame, frames=12, interval=800, repeat=True
)

gif_path = "graphrag-knowledge-graph-document-qa-graph-build.gif"
writer = PillowWriter(fps=1.2)
anim.save(gif_path, writer=writer, dpi=100)
plt.close(fig_gif)
print(f"GIF saved to: {gif_path}")
print("To optimise: gifsicle -O3 --lossy=80", gif_path, "-o", gif_path)

## Cell 6: Community Detection

We use the Louvain algorithm (via `python-louvain`) as an accessible proxy for the
Leiden algorithm used in the paper. Leiden improves on Louvain by guaranteeing
well-connected communities, but requires the `leidenalg` package.

The paper runs community detection at **multiple resolution levels** to build a hierarchy.
Here we run at a single level; the Leiden extension below shows how to vary resolution.

In [ ]:
partition = community_louvain.best_partition(G, weight="strength", random_state=42)

communities = defaultdict(list)
for node, comm_id in partition.items():
    communities[comm_id].append(node)

print(f"Communities detected: {len(communities)}")
for comm_id, members in sorted(communities.items()):
    print(f"\n  Community {comm_id} ({len(members)} nodes):")
    for m in members:
        print(f"    - {m} [{G.nodes[m].get('type', 'CONCEPT')}]")

# Modularity score (higher is better; 0.3-0.7 is typical for real networks)
modularity = community_louvain.modularity(partition, G, weight="strength")
print(f"\nModularity: {modularity:.4f}")

### Extension: Leiden with Multiple Resolution Levels

The paper uses Leiden at multiple resolution levels to create a community hierarchy.
Higher resolution = more, finer communities. Lower resolution = fewer, broader communities.

In [ ]:
# Uncomment if you have leidenalg installed: pip install leidenalg igraph
# import leidenalg
# import igraph as ig
#
# # Convert networkx graph to igraph
# ig_G = ig.Graph.from_networkx(G)
# edge_weights_ig = [G[u][v].get("strength", 0.5) for u, v in G.edges()]
#
# # Run at multiple resolution levels
# for resolution in [0.5, 1.0, 2.0]:
#     leiden_partition = leidenalg.find_partition(
#         ig_G,
#         leidenalg.CPMVertexPartition,
#         resolution_parameter=resolution,
#         weights=edge_weights_ig,
#         seed=42,
#     )
#     n_comms = len(leiden_partition)
#     print(f"Resolution {resolution:.1f}: {n_comms} communities")
print("Leiden extension code above (commented out). Install leidenalg to use.")

## Cell 7: Community Report Generation

For each community, we ask the LLM to read all entity descriptions and internal
relationships, then write a structured report: title, summary, key themes, and
most important entities.

These reports are the primary retrieval unit in GraphRAG. They are generated at
indexing time and reused across many queries.

In [ ]:
REPORT_PROMPT = """You are summarising a community of related AI/ML concepts from a knowledge graph.

Given the list of entities (with descriptions) and their relationships, write a concise community report.
Return ONLY valid JSON:
{
  "title": "...",
  "summary": "...",
  "key_themes": ["...", "..."],
  "most_important_entities": ["...", "..."]
}
"""


def generate_community_report(comm_id: int, members: list) -> dict:
    entity_lines = []
    for node in members:
        desc = G.nodes[node].get("description", "")
        entity_lines.append(
            f"- {node} ({G.nodes[node].get('type', 'CONCEPT')}): {desc}"
        )

    rel_lines = []
    for u, v, data in G.edges(data=True):
        if u in members or v in members:
            rel_lines.append(f"- {u} -> {v}: {data.get('description', '')}")

    content = "Entities:\n" + "\n".join(entity_lines)
    if rel_lines:
        content += "\n\nRelationships:\n" + "\n".join(rel_lines)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": REPORT_PROMPT},
            {"role": "user", "content": content},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    report = json.loads(response.choices[0].message.content)
    report["community_id"] = comm_id
    report["members"] = members
    return report


print("Generating community reports...")
community_reports = [
    generate_community_report(cid, members)
    for cid, members in communities.items()
]

print(f"\nGenerated {len(community_reports)} reports:\n")
for report in community_reports:
    print(f"[Community {report['community_id']}] {report['title']}")
    print(f"  Summary: {report['summary'][:120]}...")
    print(f"  Key themes: {', '.join(report['key_themes'])}")
    print(f"  Top entities: {', '.join(report['most_important_entities'][:3])}")
    print()

## Cell 8: Global Search (Map-Reduce)

Global Search is the core innovation for synthesis queries.

**Map step:** every community report is rated for relevance to the query by an LLM.
Reports below the threshold are discarded.

**Reduce step:** the surviving partial answers are synthesised into a final response.

Unlike standard RAG, Global Search processes **all** communities, guaranteeing
that no relevant information is silently missed.

In [ ]:
MAP_PROMPT = """You are analysing a community report to help answer a user query.

If this report is relevant to the query, extract the key points that address it.
Return ONLY valid JSON:
{
  "relevant": true,
  "key_points": ["...", "..."],
  "relevance_score": 0.0
}
If not relevant: {"relevant": false, "key_points": [], "relevance_score": 0.0}
"""

REDUCE_PROMPT = """You are synthesising partial answers from multiple community reports
to answer a user's question comprehensively.

Produce a well-structured, thorough answer. Avoid repetition. Cite the community titles
where relevant.
"""


def global_search(query: str, reports: list, relevance_threshold: float = 0.3) -> str:
    print(f"Global Search: '{query}'")
    print(f"Processing {len(reports)} communities...\n")

    # Map step
    relevant_points = []
    for report in reports:
        report_text = (
            f"Title: {report['title']}\n"
            f"Summary: {report['summary']}\n"
            f"Key themes: {', '.join(report['key_themes'])}\n"
            f"Important entities: {', '.join(report['most_important_entities'])}"
        )
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": MAP_PROMPT},
                {"role": "user", "content": f"Query: {query}\n\nReport:\n{report_text}"},
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )
        result = json.loads(response.choices[0].message.content)

        score = result.get("relevance_score", 0)
        is_relevant = result.get("relevant") and score > relevance_threshold

        status = "RELEVANT" if is_relevant else "skipped "
        print(f"  [{status}] {report['title']} (score={score:.2f})")

        if is_relevant:
            relevant_points.append({
                "community": report["title"],
                "points": result["key_points"],
                "score": score,
            })

    if not relevant_points:
        return "No relevant communities found for this query."

    # Reduce step
    context = "\n\n".join(
        f"From '{p['community']}':\n" + "\n".join(f"- {pt}" for pt in p["points"])
        for p in sorted(relevant_points, key=lambda x: -x["score"])
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": REDUCE_PROMPT},
            {"role": "user", "content": f"Query: {query}\n\nPartial answers:\n{context}"},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content


answer = global_search(
    "What are the main techniques for making LLMs more efficient and cost-effective?",
    community_reports,
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(textwrap.fill(answer, width=80))

## Cell 9: Local Search (Entity-Anchored Retrieval)

Local Search is for queries anchored to a specific entity.
It retrieves the entity's description, its direct neighbours in the graph,
the community report for the entity's community, and the source chunks
that mention the entity.

This is much cheaper than Global Search: a single LLM synthesis call.

In [ ]:
LOCAL_SEARCH_PROMPT = """You are a helpful assistant answering a question about specific entities
in a knowledge graph. Use the provided entity context, neighbourhood information,
community report, and source passages to give a detailed, accurate answer.
"""


def find_entity(name: str) -> str | None:
    """Case-insensitive entity lookup."""
    name_lower = name.lower()
    for node in G.nodes():
        if node.lower() == name_lower:
            return node
    # Partial match
    for node in G.nodes():
        if name_lower in node.lower():
            return node
    return None


def local_search(query: str, entity_name: str) -> str:
    node = find_entity(entity_name)
    if node is None:
        return f"Entity '{entity_name}' not found in graph. Available: {list(G.nodes())[:10]}"

    print(f"Local Search: '{query}'")
    print(f"Anchored to entity: '{node}'\n")

    # 1. Entity description
    entity_desc = G.nodes[node].get("description", "No description available.")

    # 2. Direct neighbours
    neighbours = list(G.neighbors(node))
    neighbour_info = []
    for nb in neighbours:
        edge_data = G[node][nb]
        neighbour_info.append(
            f"- {nb} [{G.nodes[nb].get('type', 'CONCEPT')}]: "
            f"{edge_data.get('description', '')} (strength={edge_data.get('strength', 0):.2f})"
        )

    # 3. Community report for this entity's community
    entity_comm_id = partition[node]
    entity_report = next(
        (r for r in community_reports if r["community_id"] == entity_comm_id), None
    )

    # 4. Source chunks mentioning this entity
    relevant_chunks = [
        chunk for chunk in CORPUS
        if node.lower() in chunk.lower()
    ]

    # Assemble context
    context_parts = [f"Entity: {node}\nDescription: {entity_desc}"]

    if neighbour_info:
        context_parts.append("Direct connections:\n" + "\n".join(neighbour_info))

    if entity_report:
        context_parts.append(
            f"Community '{entity_report['title']}':\n"
            f"{entity_report['summary']}"
        )

    if relevant_chunks:
        context_parts.append(
            "Source passages:\n" +
            "\n---\n".join(relevant_chunks[:3])
        )

    full_context = "\n\n".join(context_parts)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": LOCAL_SEARCH_PROMPT},
            {"role": "user", "content": f"Query: {query}\n\nContext:\n{full_context}"},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


local_answer = local_search(
    query="What is the KV cache and why does it matter for LLM inference?",
    entity_name="KV cache",
)

print("=" * 60)
print("LOCAL SEARCH ANSWER")
print("=" * 60)
print(textwrap.fill(local_answer, width=80))

## Cell 10: LlamaIndex GraphRAG V2 Alternative

LlamaIndex ships a `GraphRAGQueryEngine` that provides a production-ready
GraphRAG system. This is a comparison reference.

**Key differences vs the paper** (and vs our from-scratch implementation):
- Single community level only (no multi-level hierarchy)
- Global Search uses embedding lookup to *select* communities rather than
  processing all of them (reintroduces the coverage blind spot)
- Entity descriptions extracted but not used at retrieval time
- No claim extraction, no multiple gleanings

Use LlamaIndex for quick production deployment; use the from-scratch version
when you need the paper's full Global Search guarantee.

In [ ]:
# Install: pip install llama-index llama-index-packs-graph-rag-context
# (Commented out to avoid installing large dependencies by default)

# from llama_index.packs.graph_rag_context import GraphRAGQueryEngine, GraphRAGExtractor
# from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
# from llama_index.core.node_parser import SentenceSplitter
# from llama_index.llms.openai import OpenAI as LlamaOpenAI
#
# # Wrap corpus as documents
# from llama_index.core import Document
# documents = [Document(text=chunk) for chunk in CORPUS]
#
# # Chunk into nodes
# splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
# nodes = splitter.get_nodes_from_documents(documents)
#
# # Extract entities and build community graph
# llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)
# graph_extractor = GraphRAGExtractor(llm=llm, max_paths_per_chunk=2)
# graph_store = graph_extractor.extract(nodes)
#
# # Build vector index for hybrid retrieval
# index = VectorStoreIndex(nodes)
#
# # Create query engine
# query_engine = GraphRAGQueryEngine(
#     graph_store=graph_store,
#     index=index,
#     llm=llm,
#     num_children=10,
# )
#
# response = query_engine.query(
#     "What are the main techniques for making LLMs more efficient?"
# )
# print(response)

print("LlamaIndex V2 code above (commented out).")
print("Install llama-index-packs-graph-rag-context to use it.")

# Comparison summary
comparison = {
    "Feature": [
        "Community hierarchy levels",
        "Global Search mechanism",
        "Entity descriptions at retrieval",
        "Claim extraction",
        "Multiple gleanings per chunk",
        "Map-reduce relevance rating",
    ],
    "Edge et al. (2024)": [
        "Multiple (full Leiden hierarchy)",
        "True map-reduce over ALL communities",
        "Used in Local Search context",
        "Supported",
        "Yes",
        "Explicit per-community LLM rating",
    ],
    "LlamaIndex GraphRAG V2": [
        "Single level only",
        "Embedding lookup (not all communities)",
        "Extracted but not used",
        "Not implemented",
        "No",
        "Sequential aggregation",
    ],
}

print("\n" + "=" * 80)
print(f"{'Feature':<40} {'Paper':<20} {'LlamaIndex':<20}")
print("=" * 80)
for f, p, l in zip(comparison["Feature"], comparison["Edge et al. (2024)"], comparison["LlamaIndex GraphRAG V2"]):
    print(f"{f:<40} {p:<20} {l:<20}")

## Cell 11: Exercises

Try these extensions to deepen your understanding:

### Exercise 1: Multiple Gleanings
Add a gleaning loop to the extraction step. After the first extraction, prompt
the LLM: "Review the text again. Are there any entities or relationships you
missed?" Run 1-2 extra passes and compare the entity count with and without gleanings.

### Exercise 2: Vary Community Resolution
Install `leidenalg` and run community detection at three resolution levels (0.5, 1.0, 2.0).
For each level, generate community reports and run the same Global Search query.
How does the answer change across levels?

### Exercise 3: Cost Estimation
Count the total input and output tokens used by all API calls in this notebook.
Extrapolate: if your corpus had 1 million words instead of our 8 paragraphs,
what would the indexing cost be at current gpt-4o-mini pricing?

### Exercise 4: Real Corpus
Replace the hardcoded CORPUS with a set of real documents. Try:
- All arXiv abstracts from a recent ML conference
- Your own meeting notes or research papers
Compare Global Search vs standard RAG on a synthesis question.

### Exercise 5: Local vs Global
Write 5 queries. For each one, predict whether Local or Global Search would
give a better answer, then run both and compare. Does the pattern match the
paper's claim that Local Search suits entity-anchored queries?

In [ ]:
# Exercise 1 starter: gleaning loop

GLEANING_PROMPT = """Review the text again carefully.
Based on your previous extraction, are there any entities or relationships you missed?
Return ONLY valid JSON with the same structure as before:
{"entities": [...], "relationships": [...]}
If you found nothing new, return {"entities": [], "relationships": []}
"""


def extract_with_gleanings(chunk: str, n_gleanings: int = 1) -> dict:
    # First pass
    result = extract_entities_and_relationships(chunk)
    all_entities = result.get("entities", [])
    all_relationships = result.get("relationships", [])

    history = [
        {"role": "system", "content": EXTRACTION_PROMPT},
        {"role": "user", "content": f"Text:\n{chunk}"},
        {"role": "assistant", "content": json.dumps(result)},
    ]

    for gleaning_idx in range(n_gleanings):
        history.append({"role": "user", "content": GLEANING_PROMPT})
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=history,
            response_format={"type": "json_object"},
            temperature=0,
        )
        gleaning = json.loads(response.choices[0].message.content)
        history.append({"role": "assistant", "content": json.dumps(gleaning)})

        new_ents = gleaning.get("entities", [])
        new_rels = gleaning.get("relationships", [])
        all_entities.extend(new_ents)
        all_relationships.extend(new_rels)
        print(f"  Gleaning {gleaning_idx + 1}: +{len(new_ents)} entities, +{len(new_rels)} relationships")

    return {"entities": all_entities, "relationships": all_relationships}


# Test on the first chunk
print("Testing gleaning on chunk 1:")
print(f"  Baseline: {len(extractions[0].get('entities', []))} entities")
gleaned = extract_with_gleanings(CORPUS[0], n_gleanings=1)
print(f"  After 1 gleaning: {len(gleaned['entities'])} entities")